# Dia 19 — O commit apagado 🌿

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-19-git-e-github](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-19-git-e-github)

## 🎬 Missão

> Bento entra correndo na sala, notebook aberto.
> — Achei! O site da PixVelox estava num repositório público. Seis commits.
> O cara subiu um `config.env` e apagou catorze minutos depois. Já era,
> sumiu.
>
> A Mestra Iara nem tira os olhos do café.
> — Bento, o git é um diário que **não aceita borracha**. Apagar um arquivo
> é só escrever mais uma página dizendo "apaguei". As páginas de antes
> continuam lá.
>
> Antes que o repositório saísse do ar, a equipe salvou tudo num arquivo:
> `desafio/vazantes.bundle`.

**Sua tarefa:** clonar o repositório a partir do bundle, encontrar o arquivo
que maré_alta apagou e recuperar a `PAINEL_CHAVE` — a flag `VAZANTE{...}`.
De quebra, descobrir que e-mail pessoal escapou num dos commits.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Instalando e conferindo o git

```text
Git guarda o HISTÓRICO de uma pasta: cada "commit" é uma fotografia de
todos os arquivos, com autor, e-mail, data e mensagem.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "git instalado?"
precisa git
git --version

### Clonando o repositório

```text
`git clone ORIGEM PASTA` copia o repositório inteiro, COM todo o histórico.
A origem pode ser um endereço do GitHub ou, como aqui, um arquivo .bundle.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "git clone do bundle"
git bundle verify desafio/vazantes.bundle        # confere se o pacote está íntegro
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  ls -la                    # repare na pasta oculta .git: é ali que mora o histórico
  cat README.md
)
rm -rf trabalho

### git log: a linha do tempo

```text
Cada commit tem um identificador (hash). `--oneline` mostra um por linha.
`--stat` mostra quais arquivos cada commit mexeu.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "git log"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git log --oneline
  echo "---"
  git log --stat --format='%h %ad %s' --date=short
)
rm -rf trabalho

### OSINT clássico: quem são os autores?

```text
Todo commit grava NOME e E-MAIL de quem o fez — e muita gente esquece de
configurar um e-mail "de fachada" numa das máquinas.
  %an nome   %ae e-mail   %ad data   %s mensagem
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "Autores e e-mails dos commits"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git log --format='%an <%ae>' | sort | uniq -c
  echo "--- horários dos commits (hábitos do autor) ---"
  git log --format='%ad  %s' --date=format:'%d/%m %H:%M'
)
rm -rf trabalho

### git show e git diff: o que mudou?

```text
`git show HASH` mostra o commit e a diferença que ele introduziu.
`git diff A B` compara duas versões. Linhas com + entraram, com - saíram.
HEAD é o commit atual; HEAD~1 o anterior; HEAD~2 o de antes...
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "git show e git diff"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git show --stat HEAD
  echo "--- diferença do README entre o primeiro commit e agora ---"
  git diff "$(git rev-list --max-parents=0 HEAD)" HEAD -- README.md
)
rm -rf trabalho

### O truque: apagado não é esquecido

```text
Apagar um arquivo e fazer commit só tira ele do PRESENTE. Todas as versões
antigas continuam dentro do .git — e qualquer um que clonar recebe tudo.
  git log --diff-filter=D   commits que APAGARAM (Deleted) arquivos
  git log -p                mostra o conteúdo de cada mudança (patch)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "Arquivos apagados no histórico"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  ls                                             # config.env não está aqui...
  echo "--- ...mas o histórico lembra ---"
  git log --diff-filter=D --name-only --format='commit %h: %s'
  echo "--- histórico só desse arquivo, com o conteúdo (-p) ---"
  git log -p --format='commit %h %s' -- config.env | grep -v 'PAINEL_CHAVE' || true
)
rm -rf trabalho
dica "Escondemos a linha da chave de propósito. Recupere-a no desafio. 😉"

### Procurando segredos em TODO o histórico

```text
`git log -p --all` despeja todas as mudanças de todos os ramos; grep procura
padrões típicos de segredo. -S "texto" acha commits que adicionaram/removeram o texto.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "Caçando segredos"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git log -p --all | grep -E '^[+-][A-Z_]+(URL|USER|GATEWAY)=' || true
  echo "--- commits que mexeram em 'PAINEL_' ---"
  git log -S 'PAINEL_' --oneline
)
rm -rf trabalho

### Seu primeiro commit e o git status

```text
O ciclo do dia a dia:  editar -> git add -> git commit -> git push
`git status` diz o que mudou e o que já está "preparado" (staged).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "add, commit e status"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git config user.name "Recruta Coruja"                 # identidade SÓ neste repositório
  git config user.email "recruta@nucleo-coruja.example"
  ls > test.txt                                         # cria um arquivo novo
  git status --short                                    # ?? = arquivo não rastreado
  git add test.txt                                      # prepara
  git status --short                                    # A  = adicionado
  git commit -q -m "anotações do recruta"               # -m: mensagem sem abrir editor
  git log --oneline -n 2
  git status                                            # tudo limpo
)
rm -rf trabalho
interativo git commit       # sem -m o git abre um editor para a mensagem

### git push (para um "GitHub" local)

```text
`git push` envia seus commits para o repositório remoto (GitHub, GitLab...).
Para treinar sem conta, criamos um repositório "bare" (só o .git) numa pasta.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "git push para um remoto local"
mkdir -p trabalho
git init -q --bare --initial-branch=main trabalho/remoto.git   # remoto vazio, ramo principal "main"
git clone -q desafio/vazantes.bundle trabalho/pxv-site
(
  cd trabalho/pxv-site || exit 1
  git config user.name "Recruta Coruja"
  git config user.email "recruta@nucleo-coruja.example"
  git remote add coruja ../remoto.git                  # um segundo "remoto"
  echo "Relatório preliminar" > relatorio.txt
  git add relatorio.txt && git commit -q -m "relatório"
  git push -q coruja main                              # envia!
)
echo "--- o remoto recebeu: ---"
git --git-dir=trabalho/remoto.git log --oneline -n 3
rm -rf trabalho
dica "No GitHub de verdade: git push origin main (pede login/token na primeira vez)."

### No mundo real: clonando do GitHub

```text
Clonar repositório público não precisa de conta. --depth 1 baixa só a versão
mais recente (rápido), mas SEM o histórico — para OSINT você quer o histórico!
`git ls-remote` lista os ramos de um repositório sem baixar nada.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-19-git-e-github" && source "$L4O_HOME/lib/comum.sh"
titulo "GitHub de verdade"
if online && git ls-remote https://github.com/okfn-brasil/querido-diario HEAD > ls_remote.txt 2>/dev/null; then
  cat ls_remote.txt
else
  alerta "offline: só mostrando os comandos"
fi
rm -f ls_remote.txt
mostrar "git clone https://github.com/okfn-brasil/querido-diario"
mostrar "git clone --depth 1 https://github.com/okfn-brasil/querido-diario   # só o presente"

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-19-git-e-github" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-19-git-e-github" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)